# 13 — Core PyTorch API Compendium (Reference + Patterns)

Goal: cover the core `torch` surface area and how to use it correctly in real systems.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## How to use this compendium

This notebook is a **high-density map** of the PyTorch API: modules, what they do, and common usage patterns.
It is not a replacement for reading every docs page verbatim; instead it consolidates the mental model and the
most important entry points.

Major sections:
1. Tensor fundamentals & type promotion
2. Core ops: elementwise, reductions, indexing, broadcasting, out/in-place
3. Linear algebra, FFT, special functions
4. Randomness & generators
5. Serialization, checkpoints, state_dict patterns
6. Devices/backends: CUDA, MPS, CPU, pinned memory, streams
7. Debugging: anomaly detection, profiling, determinism
8. Modeling: nn.Module, containers, init, functional vs module
9. Data: datasets, dataloaders, sampling
10. Training: optimizers, schedulers, AMP, grad accumulation/checkpointing
11. Modern compilation/export stack
12. Distributed primitives (overview)
13. Interop: NumPy, DLPack, XLA notes

The goal is that you can open this notebook as a “PyTorch reference atlas”.

## 1. Type promotion, dtypes, and numerical precision

Things that matter:
- float16 vs bfloat16 vs float32
- accumulation dtypes (e.g., matmul in fp16 may accumulate in fp32 depending on backend)
- integer overflow for int32/int64
- type promotion rules for mixed dtypes

Practical defaults:
- training on CUDA: AMP with fp16 or bf16; master weights usually fp32
- inference: bf16/fp16 if supported and validated; otherwise fp32
- NLP: bf16 often more stable than fp16 on many GPUs

Key APIs:
- `torch.set_default_dtype`
- `.to(dtype=...)`, `.type_as(...)`
- `torch.autocast` (AMP)

In [ ]:

import torch
print("default dtype:", torch.get_default_dtype())
a = torch.tensor([1,2,3])        # int64 by default
b = torch.tensor([1.0,2.0,3.0])  # float32 by default
print(a.dtype, b.dtype)
print((a + b).dtype)  # promotion

## 2. In-place ops vs out-of-place ops (and autograd implications)

In-place ops (ending with `_`) can:
- improve memory usage
- but break autograd if they overwrite values needed for backward

Rule of thumb:
- use in-place ops only when you understand the autograd graph implications
- avoid in-place ops on tensors needed for gradient computation unless necessary

Examples:
- `x.add_(y)` is in-place
- `x = x + y` is out-of-place

In [ ]:

import torch
x = torch.randn(3, requires_grad=True)
y = (x * 2).sum()
# x.add_(1.0)  # would modify x in-place; may or may not error depending on graph needs
y.backward()
print("grad:", x.grad)

## 3. The `out=` parameter pattern

Many ops support `out=` to write into a preallocated tensor. This can reduce allocations in tight loops.

Example: `torch.add(a, b, out=out)`

In [ ]:

import torch
a = torch.randn(5)
b = torch.randn(5)
out = torch.empty(5)
torch.add(a, b, out=out)
out

## 4. Linear algebra (torch.linalg)

Key functions:
- solves: `solve`, `lstsq`
- decompositions: `svd`, `qr`, `cholesky`, `eig`, `eigh`
- matrix functions: `matrix_exp`
- norms: `norm`, `vector_norm`, `matrix_norm`

Use `torch.linalg` for modern, consistent linalg behavior.

In [ ]:

import torch
A = torch.randn(5,5)
A = A @ A.T + 1e-3*torch.eye(5)  # make SPD-ish
L = torch.linalg.cholesky(A)
recon = L @ L.T
print("recon error:", (A-recon).abs().max().item())

x = torch.randn(5)
sol = torch.linalg.solve(A, x)
print("solve residual:", (A@sol - x).norm().item())

## 5. FFT and signal processing

- `torch.fft.fft`, `rfft`, `fft2`, etc.
- useful in audio/vision and some physics-informed models

In [ ]:

import torch
t = torch.linspace(0, 1, 256)
sig = torch.sin(2*math.pi*10*t) + 0.5*torch.sin(2*math.pi*40*t)
spec = torch.fft.rfft(sig)
spec.abs().shape

## 6. Sparse tensors (overview)

PyTorch supports multiple sparse layouts (COO, CSR, CSC, etc.). Use sparse when:
- the tensor is truly sparse
- operations you need are supported for that sparse layout

Caveat: not all ops support sparse; verify compatibility.

In [ ]:

import torch
idx = torch.tensor([[0, 1, 1],
                    [2, 0, 2]])
vals = torch.tensor([3.0, 4.0, 5.0])
S = torch.sparse_coo_tensor(idx, vals, (2,3))
print(S)
print("dense:\n", S.to_dense())

## 7. Randomness: Generator, distributions, and determinism

- `torch.Generator` allows isolated RNG streams
- distributions: `torch.distributions` module
- determinism flags: `torch.use_deterministic_algorithms(True)` (may throw if unsupported)

For debugging: enable determinism; for production training: evaluate tradeoffs.

In [ ]:

import torch
g = torch.Generator().manual_seed(0)
print(torch.rand(3, generator=g))
print(torch.rand(3, generator=g))

## 8. Serialization: tensors, state_dict, and checkpoints

- `torch.save(obj, path)` uses pickle under the hood
- use `state_dict()` for models/optimizers
- prefer `map_location` for portability
- consider `safetensors` for safer, faster model weights in some workflows

In [ ]:

import torch, torch.nn as nn
m = nn.Linear(3,4)
sd = m.state_dict()
torch.save(sd, "linear_sd.pt")
sd2 = torch.load("linear_sd.pt", map_location="cpu")
m2 = nn.Linear(3,4)
m2.load_state_dict(sd2)
print("ok")

## 9. Device management

Common patterns:
- move model and batches to device
- pinned memory in DataLoader (CUDA)
- `non_blocking=True` for faster host→GPU copies when using pinned memory
- manage precision with AMP/autocast

## 10. Modern compilation and export stack (summary)

- `torch.compile` for speed (training/inference)
- `torch.export` for capture and transformations
- `torch.profiler` for profiling

If your goal is deployment, start with `torch.export` and validate correctness.

## 11. Quick index of core modules

- `torch`: tensor + ops + autograd
- `torch.nn`: layers + modules
- `torch.nn.functional`: stateless layer functions
- `torch.optim`: optimizers
- `torch.utils.data`: Dataset/DataLoader
- `torch.distributed`: distributed training
- `torch.cuda`: CUDA-specific utilities
- `torch.backends`: backend configs
- `torch.profiler`: performance profiling
- `torch.ao.quantization`: quantization
- `torch.fx`: graph tracing and transforms
- `torch.func`: vmap/grad/jacobians (advanced)
- `torch.export`: deployment capture

Use this as your mental table of contents.